# Emulations

This notebook demonstrates end-to-end benchmarking workflows using the `Emulator` — a drop-in QPU backend that runs a simulation in place of real hardware.

The `Emulator` ships with a default noise model:

- **Gate noise** — depolarizing channel: 0.1% on single-qubit gates, 1% on two-qubit gates.
- **Readout noise** — confusion matrix diag(0.995, 0.98) on every qubit in the config.

A custom `Simulator` can be passed via the `simulator` argument to override these defaults.

In [1]:
from qcal.backend.emulator import Emulator, _EXAMPLE_CONFIG
from qcal.benchmarking.rb import CRB
from qcal.config import Config

import logging
logging.basicConfig(level=logging.INFO)

%load_ext autoreload
%autoreload 2

## 1. Setup

Load the bundled example config and create an `Emulator` instance. The example config defines 8 qubits (0–7) with CZ connectivity.

In [2]:
config = Config(_EXAMPLE_CONFIG)
print('Qubits:', config.qubits)
print('Two-qubit pairs:', list(config.qubit_pairs))
print('Gate set:', config.native_gates['set'])

Qubits: (0, 1, 2, 3, 4, 5, 6, 7)
Two-qubit pairs: [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 0)]
Gate set: {'CZ', 'X', 'X90'}


## 2. Clifford Randomized Benchmarking (CRB)

CRB estimates the average gate fidelity by fitting an exponential decay to the survival probability of random Clifford sequences at increasing depths.

For a two-qubit pair, the default circuit depths are `[2, 4, 8, 32, 64]` (number of Cliffords per circuit, before compilation into the native gate set).

The `CRB` factory function accepts a `qpu` class (not an instance) and returns a fully-initialized protocol object. Calling `.run()` chains circuit generation, execution, analysis, and plotting.

### Two-qubit CRB on qubits (0, 1)

In [5]:
crb = CRB(
    qpu=Emulator,
    config=config,
    qubit_labels=[(0, 1)],
)
crb.run()

INFO:qcal.benchmarking.rb: Analyzing the results...



(0, 1):
Process infidelity: r = 3.46e-02 (1.39e-03) (fit with a free asymptote)
Process infidelity: r = 3.45e-02 (6.01e-04) (fit with the asymptote fixed to 1/2^n)



Runtime:   Compile  Transpile  Sequencing  Write  Measure  Process  Total
Time (s)      0.0        6.2         0.0    0.0      3.6      0.1   12.2



### Inspecting results

After `.run()`, the process infidelity and fit parameters are available as properties.

In [5]:
print('Process infidelity:', crb.process_infidelity)
print('Fit parameters:    ', crb.fit_params)

Process infidelity: {(0, 1): {'val': np.float64(0.03386475764678279), 'err': np.float64(0.0005812112536145883)}}
Fit parameters:     {(0, 1): {'base': np.float64(0.9638775918434317), 'a': np.float64(0.6671472402900825), 'b': 1, 'c': 0.25}}


## 3. CRB with a custom noise model

Pass a pre-configured `DensityMatrixSimulator` via the `simulator` argument to swap in any `ErrorModel`.

In [3]:
import numpy as np

from qcal.simulation import (
    CustomErrorModel,
    DensityMatrixSimulator,
    DepolarizingNoise,
    RelaxationNoise,
    RelaxationParams,
)

noise = CustomErrorModel(
    DepolarizingNoise(single_qubit=0.0001, two_qubit=0.001),
    RelaxationNoise(
        single_qubit=RelaxationParams(t1=50e-6, tphi=30e-6, t=50e-9),
    ),
)

p0, p1 = 0.995, 0.98
cmat = np.array([[p0, 1.0 - p0], [1.0 - p1, p1]])
for q in config.qubits:
    noise.add_readout_noise(q, cmat)

simulator = DensityMatrixSimulator(noise_model=noise, n_shots=1024)

In [4]:
crb_custom = CRB(
    qpu=Emulator,
    config=config,
    qubit_labels=[(0, 1)],
    simulator=simulator,
)
crb_custom.run()

INFO:qcal.benchmarking.rb: Analyzing the results...



(0, 1):
Process infidelity: r = 4.41e-02 (1.68e-03) (fit with a free asymptote)
Process infidelity: r = 4.48e-02 (8.10e-04) (fit with the asymptote fixed to 1/2^n)



Runtime:   Compile  Transpile  Sequencing  Write  Measure  Process  Total
Time (s)      0.0        6.4         0.0    0.0      9.2      0.2   18.5

